# MEA Model - CMU-MOSI Dataset

Multimodal fusion approach for learning modality-Exclusive and modality-Agnostic representations

In [1]:
!git clone https://github.com/ranbeer06052009/test

Cloning into 'test'...
remote: Enumerating objects: 87, done.
remote: Counting objects: 100% (87/87), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 87 (delta 15), reused 75 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (87/87), 12.61 MiB | 30.52 MiB/s, done.
Resolving deltas: 100% (15/15), done.


In [3]:
import gdown

file_id = "1szKIqO0t3Be_W91xvf6aYmsVVUa7wDHU"
destination = "mosi_raw.pkl"

gdown.download(
    f"https://drive.google.com/uc?id={file_id}", destination, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1szKIqO0t3Be_W91xvf6aYmsVVUa7wDHU
From (redirected): https://drive.google.com/uc?id=1szKIqO0t3Be_W91xvf6aYmsVVUa7wDHU&confirm=t&uuid=cfc330a2-caf0-4986-8980-e0d2a06f95d9
To: /content/mosi_raw.pkl
100%|██████████| 357M/357M [00:03<00:00, 90.6MB/s]


'mosi_raw.pkl'

In [2]:
import sys
import torch
import matplotlib.pyplot as plt

sys.path.append('/content/test/src')

from loader import get_dataloader
from models.mea import MEA
from training.train_mea import mea_criterion, train_mea_loop, test_mea
from evaluation.performance import eval_affect

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Using device:", device)

Using device: cuda


In [4]:
# Load MOSI Data
train_data, valid_data, test_data = get_dataloader(
    '/content/mosi_raw.pkl',
    max_pad=True,
    max_seq_len=50
)

In [5]:
# Initialize MEA Model with Paper Hyperparameters for MOSI
model = MEA(
    dim_l=300,   # Text dimension
    dim_v=35,    # Vision dimension
    dim_a=74,    # Audio dimension
    d=40,        # Hidden Dimension d
    dh=64,       # Output Dimension dh
    n_heads=8,   # Attention Head
    mu=0.25      # Coefficient mu
).to(device)

# Note: The paper defines learning rate 1e-3, batch size 32, epochs 60, alpha 2e-2, beta 3e-2
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [7]:
# Direct Testing without training (as requested by user)
# Note: Since the model hasn't been trained yet (no pre-trained weights loaded),
# the accuracy will be near random chance.
import numpy as np
from sklearn.metrics import f1_score

metrics, preds, labels = test_mea(
    model=model,
    dataloader=test_data,
    task_criterion=torch.nn.L1Loss(),
    device=device,
    alpha=2e-2, # Trade-off Parameter alpha
    beta=3e-2,  # Trade-off Parameter beta
    return_preds=True
)

preds_np = preds.view(-1).cpu().numpy()
labels_np = labels.view(-1).cpu().numpy()

# ===== MAE =====
mae = np.mean(np.abs(preds_np - labels_np))

# ===== Corr =====
corr = np.corrcoef(preds_np, labels_np)[0][1]
corr = 0 if np.isnan(corr) else corr

# ===== Binary =====
pred_bin = (preds_np > 0).astype(int)
label_bin = (labels_np > 0).astype(int)

mask = labels_np != 0
acc2 = (pred_bin[mask] == label_bin[mask]).mean()
f1 = f1_score(label_bin[mask], pred_bin[mask], average='weighted')

print("\nFinal Evaluation (Random Weights!):")
print(f"MAE: {mae:.4f}")
print(f"Corr: {corr:.4f}")
print(f"Acc-2: {acc2*100:.2f}%")
print(f"F1: {f1*100:.2f}%")



Final Evaluation:
MAE: 1.0535
Corr: 0.5656
Acc-2: 72.10%
F1: 72.22%
